# 店舗売上需要予測: LightGBM Baseline (Time-based Validation)

- **コンペ名:** [Store Sales - Time Series Forecasting](https://www.kaggle.com/competitions/store-sales-time-series-forecasting)
- **タスク:** エクアドルの大手スーパーFavoritaにおける店舗・商品ファミリー別 日次売上（sales）の時系列回帰予測
- **評価指標:** **RMSLE (Root Mean Squared Logarithmic Error)**
- **モデル:** **LightGBM Regressor**
- **検証設計:** 直近16日間（2017-07-31 〜 2017-08-15）を手元検証とする Time-based Split

---

## 1. 評価指標「RMSLE」と時系列の鉄則

### なぜ単なるRMSEではなく「対数誤差（RMSLE）」なのか？
売上数量は「パンの1万個」と「家電の10個」のようにスケールが大きく異なります。対数を取ることで「パーセンテージ（比率）としての誤差」を評価でき、規模の異なる商品間で公平な評価が行えます。

$$\text{RMSLE} = \sqrt{ \frac{1}{n} \sum_{i=1}^n \left( \ln(1 + \hat{y}_i) - \ln(1 + y_i) \right)^2 }$$

**実践テクニック:**
目的変数 $y$ を $z = \ln(1 + y)$ (`np.log1p`) に変換して通常のRMSE基準でLightGBMを学習させ、予測値を $\hat{y} = \exp(z) - 1$ (`np.expm1`) で元に戻すのが定石です。

### 時系列バリデーションの鉄則（未来のカンニング厳禁）
通常のランダムK-Foldは未来の情報を使って過去を当てる致命的なリークになります。手元検証はテスト期間（16日間）と全く同じ長さの **直近16日間（2017-07-31 〜 2017-08-15）** を検証データとして切り出します。

## 2. ライブラリ読み込み & データ読み込み

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

# データディレクトリの自動検出
def find_data_dir():
    candidates = [
        '/kaggle/input/store-sales-time-series-forecasting',
        '/kaggle/input/competitions/store-sales-time-series-forecasting',
        '003_store-sales/data',
        '../data',
        'data'
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, 'train.csv')):
            return c
    matches = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    if matches:
        return os.path.dirname(matches[0])
    matches_local = glob.glob('**/train.csv', recursive=True)
    if matches_local:
        return os.path.dirname(matches_local[0])
    if os.path.exists('/kaggle/input'):
        input_contents = os.listdir('/kaggle/input')
        raise FileNotFoundError(f'/kaggle/input にデータがありません。現在の中身: {input_contents}。右上の [+ Add Input] から "store-sales-time-series-forecasting" を追加してください。')
    raise FileNotFoundError('train.csv が見つかりませんでした。')

data_dir = find_data_dir()
print(f'データ読み込み元: {data_dir}')

train_df = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test.csv'))
sub_df = pd.read_csv(os.path.join(data_dir, 'sample_submission.csv'))

print(f'訓練データ形状: {train_df.shape} (期間: {train_df["date"].min()} 〜 {train_df["date"].max()})')
print(f'テストデータ形状: {test_df.shape} (期間: {test_df["date"].min()} 〜 {test_df["date"].max()})')
train_df.head()

## 3. データの探索的分析（EDA: 全体売上推移の可視化）

In [ ]:
daily_sales = train_df.groupby('date')['sales'].sum().reset_index()
daily_sales['date'] = pd.to_datetime(daily_sales['date'])

plt.figure(figsize=(14, 5))
plt.plot(daily_sales['date'], daily_sales['sales'], linewidth=1, color='#1f77b4')
plt.title('Corporación Favorita: Daily Total Sales (2013 - 2017)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Total Sales')
plt.grid(True, alpha=0.3)
# 2016年4月16日の大地震 (M7.8) を可視化
plt.axvline(pd.to_datetime('2016-04-16'), color='red', linestyle='--', alpha=0.7, label='2016-04-16 Earthquake (M7.8)')
plt.legend()
plt.tight_layout()
plt.show()

## 4. カレンダー特徴量エンジニアリング

In [ ]:
def create_calendar_features(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['dayofweek'] = df['date'].dt.dayofweek
    df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)
    df['dayofyear'] = df['date'].dt.dayofyear
    return df

train_df = create_calendar_features(train_df)
test_df = create_calendar_features(test_df)

# カテゴリカル変数の指定
categorical_cols = ['store_nbr', 'family']
for col in categorical_cols:
    train_df[col] = train_df[col].astype('category')
    test_df[col] = test_df[col].astype('category')

features = ['store_nbr', 'family', 'onpromotion', 'year', 'month', 'day', 'dayofweek', 'is_weekend', 'dayofyear']
print(f'使用特徴量 ({len(features)}列): {features}')

## 5. 時系列バリデーション分割 (Time-based Split)

In [ ]:
# 直近16日間 (2017-07-31 〜 2017-08-15) を手元検証データとする
val_start = '2017-07-31'
val_end = '2017-08-15'

# 直近のトレンドを重視し、学習データは 2016-01-01 以降を使用（高速かつ最新傾向を反映）
train_start = '2016-01-01'

trn_mask = (train_df['date'] >= train_start) & (train_df['date'] < val_start)
val_mask = (train_df['date'] >= val_start) & (train_df['date'] <= val_end)

X_train, y_train = train_df.loc[trn_mask, features], train_df.loc[trn_mask, 'sales'].values
X_val, y_val = train_df.loc[val_mask, features], train_df.loc[val_mask, 'sales'].values
X_test = test_df[features]

print(f'学習サンプル数: {len(X_train):,} 件 (期間: {train_start} 〜 2017-07-30)')
print(f'検証サンプル数: {len(X_val):,} 件 (期間: {val_start} 〜 {val_end})')

# 目的変数の対数変換 (np.log1p)
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

## 6. LightGBM 回帰モデルの学習

In [ ]:
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'feature_fraction': 0.8,
    'random_state': 42,
    'verbose': -1,
    'n_estimators': 1000,
}

model = lgb.LGBMRegressor(**params)
callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=False),
    lgb.log_evaluation(period=100)
]

model.fit(
    X_train,
    y_train_log,
    eval_set=[(X_val, y_val_log)],
    callbacks=callbacks,
)

# 特徴量重要度の可視化
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=True)
plt.figure(figsize=(10, 5))
importances.plot(kind='barh', color='#2ca02c')
plt.title('LightGBM Feature Importance (Split)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 7. 手元検証スコア（RMSLE）の評価

In [ ]:
# 検証データ予測（expm1 で対数を戻し、0未満をクリップ）
val_pred_log = model.predict(X_val)
val_pred = np.expm1(val_pred_log)
val_pred = np.clip(val_pred, 0, None)

rmsle = np.sqrt(mean_squared_error(np.log1p(y_val), np.log1p(val_pred)))
print('=' * 50)
print(f'  🏆 手元検証スコア (Validation RMSLE): {rmsle:.5f}')
print('=' * 50)

# 検証期間の実測値 vs 予測値（総売上合計）の比較
val_df = train_df.loc[val_mask].copy()
val_df['pred_sales'] = val_pred
daily_val = val_df.groupby('date')[['sales', 'pred_sales']].sum().reset_index()

plt.figure(figsize=(12, 4))
plt.plot(daily_val['date'], daily_val['sales'], label='Actual (y_true)', marker='o', color='#1f77b4')
plt.plot(daily_val['date'], daily_val['pred_sales'], label='Predicted (y_pred)', marker='x', linestyle='--', color='#ff7f0e')
plt.title(f'Validation Period Daily Total Sales: Actual vs Predicted (RMSLE: {rmsle:.5f})')
plt.xlabel('Date')
plt.ylabel('Total Sales')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 8. テストデータ予測 & 提出ファイル作成

In [ ]:
test_pred_log = model.predict(X_test)
test_pred = np.expm1(test_pred_log)
test_pred = np.clip(test_pred, 0, None)

sub_df['sales'] = test_pred
sub_df.to_csv('submission.csv', index=False)

print('提出用 submission.csv を作成しました！')
print(f'行数: {len(sub_df):,} 行')
print(sub_df.head(10))
print(sub_df.describe())